# Глава 11. Нейросети — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Нейрон и слои

Нейросеть звучит загадочно, но её кирпичик вам уже знаком: взвешенная сумма признаков плюс нелинейная «функция активации» — почти логистическая регрессия. Собираем нейрон и маленькую сеть на numpy и разбираемся, зачем нужны слои.

Теория — на сайте курса: `ch11/01-neuron.html`. Задачи «собери код из строк» (1) остались на сайте.

### Один нейрон · разминка

Напишите функцию `neuron(x, w, b)`, которая возвращает сигмоиду от `x @ w + b`. Сохраните `p` — выход нейрона для гостя `x` с весами `w` и `b` (3 знака).

*Подсказки (2): `dsc.hint('neuron')`*

In [ ]:
%%task ch11/01-neuron/neuron
import numpy as np
x = np.array([30.0, 0.0, 1.0])
w = np.array([0.05, 1.5, -0.8])
b = -1.0

### ReLU и её производная · разминка

Сохраните `r = relu(z)` для вектора `z` и `dr` — «производную» ReLU: 1, где z > 0, и 0 в остальных местах (массив целых чисел).

*Подсказки (2): `dsc.hint('relu')`*

In [ ]:
%%task ch11/01-neuron/relu
import numpy as np
z = np.array([-1.5, 0.0, 0.3, 2.0, -0.2])

### Сколько параметров · разминка

Напишите `n_params(sizes)` — число параметров (веса + сдвиги) полносвязной сети со слоями заданных размеров, как в разборе урока. Посчитайте `p_small` для `[10, 16, 8, 1]`, `p_logit` для `[10, 1]` и `p_big` для `[10, 256, 256, 1]`.

*Подсказки (2): `dsc.hint('n-params')`*

In [ ]:
%%task ch11/01-neuron/n-params
def n_params(sizes):
    ...

### Слои без активации · база

Покажите, что два линейных слоя без активации — один линейный слой. Для `X`, `W1`, `W2` из заготовки посчитайте `two_layers = (X @ W1) @ W2`, найдите матрицу `W` одного слоя, дающего то же самое, и сохраните её. В `same` — `True`, если `X @ W` совпадает с `two_layers`. В `with_relu_same` — совпадёт ли `np.maximum(0, X @ W1) @ W2` с `X @ W`.

*Подсказки (2): `dsc.hint('collapse')`*

In [ ]:
%%task ch11/01-neuron/collapse
import numpy as np
rng = np.random.default_rng(1)
X = rng.normal(size=(5, 3))
W1 = rng.normal(size=(3, 4))
W2 = rng.normal(size=(4, 2))

### Прямой проход сети · база

Напишите функцию `forward(X, W1, b1, w2, b2)` для сети «скрытый слой с ReLU → один выходной нейрон с сигмоидой», возвращающую вероятности. Посчитайте `p` для `X` и весов из заготовки (3 знака) и `pred` — классы при пороге 0.5.

*Подсказки (2): `dsc.hint('forward')`*

In [ ]:
%%task ch11/01-neuron/forward
import numpy as np
X = np.array([[0.2, 1.0], [1.5, -0.3], [-1.0, 0.4]])
W1 = np.array([[1.0, -2.0, 0.5], [0.3, 1.0, -1.0]])
b1 = np.array([0.0, 0.1, 0.2])
w2 = np.array([1.5, -1.0, 2.0])
b2 = -0.5

### Где активация «засыпает» · база

Обучение сети двигает веса пропорционально наклону (производной) активации. Посчитайте производные на сетке `z = np.linspace(-6, 6, 121)`: сигмоиды `σ(z)·(1 − σ(z))`, tanh `1 − tanh(z)²` и ReLU (1 при z > 0, иначе 0). Сохраните `max_deriv` — словарь `{"sigmoid": …, "tanh": …}` наибольших производных (3 знака), `deriv_at_5` — словарь производных трёх функций в точке z = 5 (4 знака; ключи `"sigmoid"`, `"tanh"`, `"relu"`) и `share_small` — долю точек сетки, где производная сигмоиды меньше 0.01 (3 знака).

*Подсказки (3): `dsc.hint('saturation')`*

In [ ]:
%%task ch11/01-neuron/saturation
import numpy as np

z = np.linspace(-6, 6, 121)

### Формы по слоям · посложнее

Пакет из 100 гостей с 10 признаками проходит через сеть со слоями `[10, 16, 8, 1]` (ReLU в скрытых слоях, сигмоида на выходе). Веса — в заготовке (`Ws`, `bs` — списки). Напишите `forward(X, Ws, bs)`, возвращающий выход сети и список форм после каждого слоя. Сохраните `out_shape` — форму выхода, `shapes` — список форм (кортежей), `p_mean` — среднюю вероятность (3 знака). Затем выясните, что будет, если перепутать порядок весов: вызов `forward(X, Ws[::-1], bs[::-1])` падает — сохраните в `err_type` имя типа ошибки.

*Подсказки (3): `dsc.hint('shapes')`*

In [ ]:
%%task ch11/01-neuron/shapes
import numpy as np

# X (100 × 10), Ws и bs (списки весов и сдвигов по слоям) уже созданы
def forward(X, Ws, bs):
    shapes = []
    A = X
    ...
    return A, shapes

### Где не справляется прямая · посложнее

Проверьте, что логистическая регрессия не решает задачу «ровно одно из двух», а сеть из теории решает. Сохраните `acc_logit` — долю правильных ответов `LogisticRegression()` на 4 точках `X`, `y` (обучение и проверка на них же) и `acc_net` — долю правильных ответов сети с весами из теории при пороге 0.5.

*Подсказки (2): `dsc.hint('xor-linear')`*

In [ ]:
%%task ch11/01-neuron/xor-linear
import numpy as np
from sklearn.linear_model import LogisticRegression
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W1 = np.array([[20.0, 20.0], [20.0, 20.0]])
b1 = np.array([-10.0, -30.0])
w2 = np.array([20.0, -40.0])
b2 = -10.0

### Скрытый слой как новые признаки · со звёздочкой

Сеть из теории можно понять как логистическую регрессию над новыми признаками. Посчитайте скрытый слой `H = sigmoid(X @ W1 + b1)` (данные «крест» из прошлой задачи) и обучите `LogisticRegression()` на `H` вместо `X`. Сохраните `H` (округлите до 2 знаков) и `acc_on_h` — долю правильных ответов на тех же 4 точках.

*Подсказки (2): `dsc.hint('feature-net')` · задача по желанию*

In [ ]:
%%task ch11/01-neuron/feature-net
import numpy as np
from sklearn.linear_model import LogisticRegression
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W1 = np.array([[20.0, 20.0], [20.0, 20.0]])
b1 = np.array([-10.0, -30.0])

## Урок 2. Обучение сети: обратное распространение ошибки

Как сеть сама находит веса всех слоёв? Тем же градиентным спуском из главы 10. Новое только одно: градиенты для внутренних слоёв считаются «с конца» — от ошибки на выходе назад к входу. Обучаем двухслойную сеть на numpy и смотрим, как она изгибает границу классов.

Теория — на сайте курса: `ch11/02-backprop.html`. Задачи «собери код из строк» (1) остались на сайте.

### Логистическая потеря · разминка

Напишите функцию `logloss(y, p)` и сохраните потери для двух моделей: `l_good` — уверенной и правой, `l_bad` — уверенной и неправой (3 знака).

*Подсказки (2): `dsc.hint('logloss')`*

In [ ]:
%%task ch11/02-backprop/logloss
import numpy as np
y = np.array([1, 0, 1, 1])
p_good = np.array([0.9, 0.1, 0.8, 0.95])
p_bad = np.array([0.1, 0.9, 0.2, 0.05])

### Проверка формулы p − y · разминка

Проверьте численно, что производная логистической потери по оценке z для одного объекта равна p − y. Для `y = 1` и `z = 0.4` посчитайте `analytic = sigmoid(z) - y` и `numeric` — численную производную функции `L(z) = -log(sigmoid(z))` с `eps = 1e-5` (оба 5 знаков).

*Подсказки (2): `dsc.hint('grad-check')`*

In [ ]:
%%task ch11/02-backprop/grad-check
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

y, z, eps = 1, 0.4, 1e-5

### Цепочка руками · база

Повторите разбор маленькой сети из урока для другого гостя: x = 2, w1 = 0.3, w2 = −1, гость **остался** (y = 0). Посчитайте прямой проход и градиенты: `p` — вероятность ухода, `d2` — p − y, `g_w2` — градиент по w2, `g_w1` — градиент по w1 (все — 4 знака). В `w1_new`, `w2_new` — веса после одного шага спуска с lr = 0.5 (4 знака).

*Подсказки (3): `dsc.hint('ch11/02-backprop/chain')`*

In [ ]:
%%task ch11/02-backprop/chain
import numpy as np

x, w1, w2, y = 2.0, 0.3, -1.0, 0

### Обучить сеть · база

Напишите функцию `train(X, y, hidden, lr, steps, seed)`, которая обучает сеть из теории и возвращает кортеж `(W1, b1, w2, b2)`. Инициализация — как в теории: `rng = np.random.default_rng(seed)`, сначала `W1 = rng.normal(size=(2, hidden)) * 0.5`, потом `w2 = rng.normal(size=hidden) * 0.5`, смещения — нули. Обучите на полумесяцах (`hidden=8, lr=0.5, steps=1500, seed=0`) и сохраните `acc` — долю правильных ответов на этих данных (3 знака).

*Подсказки (2): `dsc.hint('train-net')`*

In [ ]:
%%task ch11/02-backprop/train-net
import numpy as np
from sklearn.datasets import make_moons
X, y = make_moons(400, noise=0.2, random_state=0)

### Шаг обучения · база

Функция `train(lr, steps)` из заготовки обучает сеть из урока (16 нейронов, полумесяцы, seed 0) и возвращает итоговую потерю и долю верных. Посчитайте `res` — словарь `{lr: (потеря, доля верных)}` для `lr` из `[0.05, 0.5, 5, 20]` при 2000 шагах (3 знака). Сохраните `best_lr` — шаг с наименьшей потерей и `broken` — список шагов, при которых доля верных ниже 0.85.

*Подсказки (2): `dsc.hint('lr')`*

In [ ]:
%%task ch11/02-backprop/lr
res = {}

### Почему не нули · база

Обучите сеть функцией `train` из прошлой задачи (она уже в заготовке), но замените инициализацию: все веса — нули, кроме `w2`, который заполните одним и тем же числом 0.1 (`np.full(hidden, 0.1)`). Для этого в заготовке есть функция `train_const`. Сохраните `acc_const` — долю правильных ответов (3 знака), и `same_neurons` — `True`, если после обучения все столбцы `W1` одинаковые (`np.allclose` каждого столбца с первым).

*Подсказки (2): `dsc.hint('zero-init')`*

In [ ]:
%%task ch11/02-backprop/zero-init
import numpy as np
from sklearn.datasets import make_moons
X, y = make_moons(400, noise=0.2, random_state=0)

def train_const(X, y, hidden, lr, steps):
    W1 = np.zeros((2, hidden))
    b1 = np.zeros(hidden)
    w2 = np.full(hidden, 0.1)
    b2 = 0.0
    for _ in range(steps):
        Z1 = X @ W1 + b1
        H = np.maximum(0, Z1)
        p = 1 / (1 + np.exp(-(H @ w2 + b2)))
        d2 = (p - y) / len(y)
        dH = np.outer(d2, w2) * (Z1 >= 0)          # >= 0, иначе при нулях сигнал вообще не пройдёт
        w2 = w2 - lr * H.T @ d2
        b2 = b2 - lr * d2.sum()
        W1 = W1 - lr * X.T @ dH
        b1 = b1 - lr * dH.sum(axis=0)
    return W1, b1, w2, b2

### Сколько нейронов · посложнее

С помощью `train` (в заготовке) обучите сети с `hidden` = 1, 2, 4, 16, 64 (`lr=0.5, steps=2000, seed=0`) на обучающей части полумесяцев и проверьте долю правильных ответов на тестовой. Данные: `make_moons(600, noise=0.3, random_state=1)`, `train_test_split(X, y, test_size=0.5, random_state=0)`. Сохраните словарь `acc_test` (`{hidden: доля}`, 3 знака) и `best_hidden` (при равенстве — меньший).

*Подсказки (2): `dsc.hint('hidden-size')`*

In [ ]:
%%task ch11/02-backprop/hidden-size
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

def train(X, y, hidden, lr, steps, seed):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=(X.shape[1], hidden)) * 0.5
    b1 = np.zeros(hidden)
    w2 = rng.normal(size=hidden) * 0.5
    b2 = 0.0
    for _ in range(steps):
        Z1 = X @ W1 + b1
        H = np.maximum(0, Z1)
        p = 1 / (1 + np.exp(-(H @ w2 + b2)))
        d2 = (p - y) / len(y)
        dH = np.outer(d2, w2) * (Z1 > 0)
        w2 = w2 - lr * H.T @ d2
        b2 = b2 - lr * d2.sum()
        W1 = W1 - lr * X.T @ dH
        b1 = b1 - lr * dH.sum(axis=0)
    return W1, b1, w2, b2

def predict(params, X):
    W1, b1, w2, b2 = params
    return 1 / (1 + np.exp(-(np.maximum(0, X @ W1 + b1) @ w2 + b2)))

X, y = make_moons(600, noise=0.3, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

### Проверка обратного прохода · со звёздочкой

Убедитесь, что формулы обратного прохода верны. Для маленькой сети из заготовки посчитайте аналитический градиент потери по элементу `W1[0, 1]` (по формулам из теории) и численный — сдвигая этот элемент на ±1e-5 и пересчитывая потерю. Сохраните `g_analytic`, `g_numeric` (6 знаков) и `rel_err` — `|a − n| / (|a| + |n|)`.

*Подсказки (2): `dsc.hint('backprop-check')` · задача по желанию*

In [ ]:
%%task ch11/02-backprop/backprop-check
import numpy as np
rng = np.random.default_rng(3)
X = rng.normal(size=(20, 2))
y = (X[:, 0] * X[:, 1] > 0).astype(float)
W1 = rng.normal(size=(2, 4)) * 0.5
b1 = np.zeros(4)
w2 = rng.normal(size=4) * 0.5
b2 = 0.0

def loss(W1):
    p = 1 / (1 + np.exp(-(np.maximum(0, X @ W1 + b1) @ w2 + b2)))
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

## Урок 3. Граф вычислений, инициализация и обрезка градиента

Обратный проход для двухслойной сети мы вывели руками. Теперь разбираем, как его считают библиотеки — графом вычислений, — и что нужно, чтобы глубокая сеть вообще училась: правильный масштаб начальных весов (Xavier и He) и обрезка градиента, которая спасает от редких огромных шагов.

Теория — на сайте курса: `ch11/02b-graph.html`.

### Граф вычислений своими руками · база

С помощью класса `Value` из заготовки (тот же, что в теории) постройте граф потери $L = (w \cdot x + b - y)^2$ для `x = 1.5`, `w = -2`, `b = 4`, `y = 3` и вызовите `L.backward()`. Сохраните:

- `loss` — значение `L.data`;
- `grads` — словарь `{"w": …, "b": …, "x": …}` с градиентами из графа;
- `manual_w` — градиент по `w`, посчитанный по формуле $2 \cdot (wx + b - y) \cdot x$ обычными числами, и `same` — `True`, если он совпал с `grads["w"]`;
- `g_twice` — градиент функции $f = w \cdot x + w \cdot w$ по `w` в той же точке (`w = -2`, `x = 1.5`). Для неё заведите **новые** объекты `Value`.

*Подсказки (2): `dsc.hint('graph')`*

In [ ]:
%%task ch11/02b-graph/graph
class Value:
    def __init__(self, data, parents=(), local_grads=()):
        self.data = data
        self.grad = 0.0
        self.parents = parents
        self.local_grads = local_grads

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), (1.0, 1.0))

    def __sub__(self, other):
        return Value(self.data - other.data, (self, other), (1.0, -1.0))

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v.parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            for p, g in zip(v.parents, v.local_grads):
                p.grad += v.grad * g

### Масштаб начальных весов · база

Сравните четыре способа задать начальные веса в сети из 8 слоёв по 64 нейрона с ReLU. Данные `X` (256 объектов × 64 признака) уже в заготовке. Напишите функцию `last_layer_std(scale)`: она заводит генератор `np.random.default_rng(1)`, 8 раз подряд делает `A = np.maximum(0, A @ (r.normal(size=(64, 64)) * scale))`, начиная с `A = X`, и возвращает `A.std()` после последнего слоя. Сохраните:

- `last_std` — словарь `{"1.0": …, "0.01": …, "xavier": …, "he": …}` с разбросом после 8-го слоя для масштабов 1.0, 0.01, $\sqrt{1/64}$ и $\sqrt{2/64}$, каждый — с тремя значащими цифрами: `float(f"{v:.3g}")`;
- `best` — ключ, у которого разброс ближе всего к 1 в «разах»: наименьшее `abs(np.log10(std))`.

*Подсказки (3): `dsc.hint('init-scale')`*

In [ ]:
%%task ch11/02b-graph/init-scale
import numpy as np
X = np.random.default_rng(0).normal(size=(256, 64))

def last_layer_std(scale, layers=8, width=64):
    r = np.random.default_rng(1)
    A = X
    ...

### Обрезка по норме · база

Напишите функцию `clip_by_norm(g, max_norm)`: если норма вектора `g` (`np.linalg.norm`) больше `max_norm`, она возвращает `g`, умноженный на `max_norm / норма`, иначе — `g` без изменений. Сохраните `a` — результат для `[30.0, 40.0]` с порогом 5, `b` — для `[0.3, 0.4]` с порогом 5, `c` — для `[3.0, -12.0, 4.0]` с порогом 6.5 и `norm_c` — норму `c` (2 знака).

*Подсказки (3): `dsc.hint('clip-norm')`*

In [ ]:
%%task ch11/02b-graph/clip-norm
import numpy as np

def clip_by_norm(g, max_norm):
    ...

### Порог обрезки · посложнее

Данные доставок с пятью ошибочными записями уже подготовлены (`X`, `y`, `normal` — как в теории). В заготовке — функция `sgd` из теории без обрезки. Допишите её: если задан `max_norm` и норма градиента больше порога, обрежьте градиент по норме, а в `clipped` считайте, сколько раз это случилось. Функция должна возвращать `(mae, norms, clipped)`.

Запустите её с `lr=0.1` для порогов `None` (без обрезки), 50, 20 и 5. Сохраните:

- `res` — словарь `{порог: (худшая MAE во второй половине шагов, MAE после последнего шага, доля шагов с обрезкой)}`, MAE — 2 знака, доля — 3 знака; «вторая половина» — `mae[len(mae) // 2:]`;
- `safest` — порог с наименьшей худшей MAE. «Безопаснее» по этой метрике ещё не значит «лучше»: что не так с самым низким порогом, смотрите в разборе;
- `median_norm` — медиану `norms` без обрезки (1 знак).

*Подсказки (3): `dsc.hint('explode')`*

In [ ]:
%%task ch11/02b-graph/explode
import numpy as np

def sgd(lr, max_norm=None, epochs=5, batch=32):
    rng = np.random.default_rng(0)
    w, b = np.zeros(2), 0.0
    mae, norms, clipped = [], [], 0
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for start in range(0, len(y), batch):
            i = order[start:start + batch]
            err = X[i] @ w + b - y[i]
            g = np.append(2 * X[i].T @ err / len(i), 2 * err.mean())
            norms.append(np.linalg.norm(g))
            # здесь — обрезка по норме и подсчёт clipped
            w, b = w - lr * g[:2], b - lr * g[2]
            mae.append(np.abs(X[normal] @ w + b - y[normal]).mean())
    return np.array(mae), np.array(norms), clipped

res = {}

## Урок 4. Нейросеть в sklearn: MLP

Писать обратный проход руками для каждой задачи не нужно. В sklearn есть многослойный перцептрон — MLPClassifier и MLPRegressor. Учимся задавать архитектуру, регуляризацию и раннюю остановку, смотрим на кривую обучения и честно сравниваем сеть с бустингом на табличных данных.

Теория — на сайте курса: `ch11/03-mlp.html`. Задачи «собери код из строк» (1) остались на сайте.

### Первая сеть · разминка

Обучите `make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=300, random_state=0))` на оттоке и сохраните `auc` на тесте (3 знака) и `n_weights` — число весов в первом слое (`model[-1].coefs_[0].size`).

*Подсказки (2): `dsc.hint('first-mlp')`*

In [ ]:
%%task ch11/03-mlp/first-mlp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Сколько параметров · разминка

Напишите функцию `n_params(n_in, layers, n_out)`, которая считает число параметров (весов и смещений) полносвязной сети: вход `n_in` признаков, скрытые слои `layers` (кортеж), выход `n_out` нейронов. Сохраните `p1 = n_params(15, (32,), 1)` — сеть из раздела про `alpha` — и `p2 = n_params(64, (128, 64), 10)`.

*Подсказки (2): `dsc.hint('count-params')`*

In [ ]:
%%task ch11/03-mlp/count-params
# ваш код

### Подбор регуляризации · база

Для `alpha` в `[0.0001, 0.1, 1.0, 10.0]` обучите `make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), alpha=..., max_iter=500, random_state=0))` и посчитайте AUC на обучении и на тесте. Сохраните DataFrame `res` (столбцы `alpha`, `train`, `test`, 3 знака), `best_alpha` — наибольший AUC на тесте (при равенстве — меньший alpha), и `gap_small`, `gap_big` — разрыв `train − test` при alpha 0.0001 и 10 (3 знака).

*Подсказки (2): `dsc.hint('alpha-sweep')`*

In [ ]:
%%task ch11/03-mlp/alpha-sweep
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Кривая потерь · база

Обучите `MLPClassifier(hidden_layer_sizes=(32,), max_iter=200, random_state=0)` (в конвейере со StandardScaler) и сохраните `curve` — список `loss_curve_`, `first`, `last` — потери в первую и последнюю эпоху (3 знака) и `halved_at` — номер первой эпохи (считая с 1), где потеря стала меньше половины первой.

*Подсказки (2): `dsc.hint('loss-curve')`*

In [ ]:
%%task ch11/03-mlp/loss-curve
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Разрыв по эпохам · база

Повторите разбор из урока (сеть `MLPClassifier(hidden_layer_sizes=(128,), alpha=0.0001, random_state=0)`, обучение по одной эпохе через `partial_fit` на масштабированных данных из заготовки) на 150 эпохах и сохраните после **каждой** эпохи AUC на обучении и тесте: списки `auc_train`, `auc_test` (длины 150, без округления). Сохраните `best_epoch` — номер эпохи (от 1) с наибольшим AUC на тесте, `best_test` (3 знака) и `gap_end` — разрыв «обучение − тест» после последней эпохи (3 знака).

*Подсказки (2): `dsc.hint('epoch-gap')`*

In [ ]:
%%task ch11/03-mlp/epoch-gap
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

net = MLPClassifier(hidden_layer_sizes=(128,), alpha=0.0001, random_state=0)
auc_train, auc_test = [], []

### Шире или глубже · посложнее

Сравните на оттоке три архитектуры при `alpha=1.0`, `max_iter=500`, `random_state=0` (конвейер со `StandardScaler`): `(8,)`, `(32,)`, `(64, 32)`. Сохраните `res` — словарь `{архитектура-строка: AUC на тесте}` с ключами `"8"`, `"32"`, `"64-32"` (3 знака), `params` — словарь с числом параметров каждой сети (сумма размеров `coefs_` и `intercepts_`), и `best` — ключ лучшей.

*Подсказки (2): `dsc.hint('ch11/03-mlp/depth')`*

In [ ]:
%%task ch11/03-mlp/depth
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

archs = {"8": (8,), "32": (32,), "64-32": (64, 32)}
res, params = {}, {}

### Сеть для минут доставки · посложнее

Обучите `make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32,), max_iter=2000, random_state=0))` на доставках (признаки — как в главе 10: `distance_km`, `items`, `hour`, флаги `courier`, `weather`, `shop_id`) и сравните с `HistGradientBoostingRegressor(random_state=0)`. Сеть использует все 2000 эпох, поэтому предупреждение `ConvergenceWarning` ожидаемо, а обучение в браузере займёт заметное время — от нескольких секунд до десятков. Сохраните `mae_mlp`, `mae_hgb` (2 знака) и `winner` — `"сеть"` или `"бустинг"`.

*Подсказки (2): `dsc.hint('regression-net')`*

In [ ]:
%%task ch11/03-mlp/regression-net
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = pd.get_dummies(ok[["distance_km", "items", "hour", "courier", "weather", "shop_id"]], dtype=int)
y = ok["actual_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Архитектура кросс-валидацией · со звёздочкой

Подберите архитектуру и регуляризацию сети для оттока: `GridSearchCV` по конвейеру `make_pipeline(StandardScaler(), MLPClassifier(max_iter=300, random_state=0))` с сеткой `{"mlpclassifier__hidden_layer_sizes": [(16,), (32, 16)], "mlpclassifier__alpha": [0.1, 1.0]}`, `cv=3`, `scoring="roc_auc"`. Сохраните `best_params`, `cv_auc` и `test_auc` (3 знака).

*Подсказки (2): `dsc.hint('arch-grid')` · задача по желанию*

In [ ]:
%%task ch11/03-mlp/arch-grid
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 5. PyTorch: то же самое, но в промышленном масштабе

Нейросети на практике пишут на PyTorch. Он делает за вас самое трудоёмкое из обратного прохода — считает градиенты автоматически — и умеет обучать сети на видеокарте. Разбираем, как наш numpy-код превращается в PyTorch: тензоры, автоматические градиенты, класс сети `nn.Module`, цикл обучения и сохранение весов.

Теория — на сайте курса: `ch11/04-pytorch.html`. Задачи «собери код из строк» (1) остались на сайте.

### float64 → float32 · разминка

PyTorch по умолчанию работает во float32, а numpy и pandas отдают float64. Посмотрите, что меняется при переходе. В заготовке — признаки гостей `X64`. Сохраните:

- `X32` — та же матрица во float32 (`astype`);
- `ratio` — во сколько раз `X64` занимает больше байт, чем `X32` (`.nbytes`);
- `max_err` — наибольшее расхождение `|X64 − X32|` по всем клеткам, две значащие цифры: `float(f"{v:.2g}")`;
- `revenue32` — годовая выручка 123 456 789.12 рубля, записанная во float32 и возвращённая в обычный `float`, и `lost_rub` — на сколько рублей она отличается от исходной (2 знака).

*Подсказки (2): `dsc.hint('float32')`*

In [ ]:
%%task ch11/04-pytorch/float32
import numpy as np
import pandas as pd
g = pd.read_csv("guests_churn.csv")
X64 = g[["tenure_months", "visits_90d", "days_since_last", "delivery_share"]].to_numpy(float)

### Сеть классом на numpy · база

Напишите класс `TinyNet` — сеть произвольной глубины в стиле `nn.Module`:

- `__init__(self, sizes, seed=0)` — `sizes` вроде `[4, 16, 8, 1]`; генератор `rng = np.random.default_rng(seed)` создаётся один раз, и для каждой пары соседних размеров `(a, b)` по порядку создаются веса `rng.normal(size=(a, b)) * np.sqrt(2 / a)` (He) и нулевые сдвиги длины `b`; храните их в списках `self.Ws` и `self.bs`;
- `forward(self, X)` — ReLU во всех слоях, кроме последнего; последний слой без активации; вернуть логиты формы `(число объектов,)`;
- `n_params(self)` — общее число весов и сдвигов.

Создайте `net = TinyNet([4, 16, 8, 1])` и примените к первым пяти гостям из стандартизованной матрицы `X` (она в заготовке). Сохраните `n_par`, `out_shape` — форму выхода и `p` — вероятности ухода этих гостей (сигмоида от логитов, 3 знака).

*Подсказки (3): `dsc.hint('module-class')`*

In [ ]:
%%task ch11/04-pytorch/module-class
import numpy as np
import pandas as pd
g = pd.read_csv("guests_churn.csv")
X = g[["tenure_months", "visits_90d", "days_since_last", "complaints"]].to_numpy(float)
X = (X - X.mean(axis=0)) / X.std(axis=0)

class TinyNet:
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.Ws, self.bs = [], []
        ...

    def forward(self, X):
        ...

    def n_params(self):
        ...

## Урок 6. Softmax, выход под задачу и мини-батчи

Сеть умеет отвечать «да/нет». Теперь учим её выбирать один класс из многих — softmax и кросс-энтропия, — разбираемся, какой выход и какую потерю брать под любую задачу «Зерна», и нарезаем данные на мини-батчи своими руками. Это всё понадобится в проекте главы.

Теория — на сайте курса: `ch11/04b-softmax.html`.

### Softmax · разминка

Напишите функцию `softmax(Z)` для матрицы логитов (строка — объект) со сдвигом на максимум строки. Сохраните `P = softmax(Z)` и `pred` — номер класса с наибольшей вероятностью для каждой строки.

*Подсказки (2): `dsc.hint('ch11/04b-softmax/softmax')`*

In [ ]:
%%task ch11/04b-softmax/softmax
import numpy as np
Z = np.array([[1.0, 3.0, 0.5],
              [2.0, 2.0, 4.0],
              [1000.0, 999.0, 998.0]])

### Кросс-энтропия · разминка

Напишите функцию `cross_entropy(P, y)` — средний −log вероятности правильного класса. Сохраните `ce` для `P` и `y` из заготовки (4 знака) и `worst` — номер объекта с наибольшим штрафом.

*Подсказки (2): `dsc.hint('cross-entropy')`*

In [ ]:
%%task ch11/04b-softmax/cross-entropy
import numpy as np
P = np.array([[0.7, 0.2, 0.1],
              [0.1, 0.1, 0.8],
              [0.3, 0.6, 0.1],
              [0.05, 0.9, 0.05]])
y = np.array([0, 2, 0, 1])

### One-hot для классов · база

Для обучения сети на K классах ответ иногда нужен в виде **one-hot** — матрицы, где в строке объекта стоит 1 в столбце его класса и 0 в остальных. Напишите функцию `one_hot(y, k)` и сохраните `Y = one_hot(y, 4)`. Проверьте полезное свойство: кросс-энтропия = −mean(sum(Y · log P, по строке)) — сохраните `ce_onehot` для `P` из заготовки (4 знака) и `same` — `True`, если она совпадает с вычисленной через `P[np.arange(n), y]`.

*Подсказки (2): `dsc.hint('one-hot')`*

In [ ]:
%%task ch11/04b-softmax/one-hot
import numpy as np
y = np.array([2, 0, 3, 1, 2])
P = np.array([[0.1, 0.1, 0.7, 0.1],
              [0.6, 0.2, 0.1, 0.1],
              [0.2, 0.2, 0.2, 0.4],
              [0.3, 0.4, 0.2, 0.1],
              [0.25, 0.25, 0.25, 0.25]])

### Температура softmax · база

Если разделить логиты на число T перед softmax, распределение становится острее (T < 1) или ровнее (T > 1) — этот приём называют **температурой** (его используют и в языковых моделях, глава 13). Для логитов `[2.0, 1.0, 0.1]` и T из `[0.5, 1, 2, 5]` посчитайте вероятности и **энтропию** распределения `−Σ p·log p`. Сохраните `top_prob` — словарь `{T: вероятность первого класса}` и `entropy` — словарь `{T: энтропия}` (оба 3 знака), и `max_entropy` — энтропию равномерного распределения на 3 класса (3 знака).

*Подсказки (2): `dsc.hint('temperature')`*

In [ ]:
%%task ch11/04b-softmax/temperature
import numpy as np

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

logits = np.array([2.0, 1.0, 0.1])

### Три головы — три потери · база

Сеть «Зерна» с общими скрытыми слоями и тремя «головами» выдала сырые выходы (логиты) для четырёх гостей:

- `z_reg` — прогноз минут доставки, правда — `y_reg`;
- `z_bin` — логит «уйдёт ли гость», правда — `y_bin`;
- `Z_ml` — три логита «купит ли кофе, десерт, сэндвич», правда — `Y_ml` (гость может купить что угодно вместе).

Сохраните словарь `losses` с ключами `"regression"` (MSE), `"binary"` (логистическая потеря) и `"multilabel"` (логистическая потеря, усреднённая по всем 12 клеткам), все — 3 знака. Логистическую потерю считайте устойчивой формулой по логитам: `np.logaddexp(0, z) - y * z`. В `n_cats` — список: сколько категорий у каждого гостя получили вероятность больше 0.5.

*Подсказки (3): `dsc.hint('head-loss')`*

In [ ]:
%%task ch11/04b-softmax/head-loss
import numpy as np
z_reg = np.array([31.5, 44.0, 27.0, 52.0]); y_reg = np.array([30.0, 47.0, 27.0, 40.0])
z_bin = np.array([1.2, -0.4, 3.0, -2.5]);  y_bin = np.array([1, 1, 1, 0])
Z_ml = np.array([[2.0, -0.5, -3.0],
                 [0.3, 1.5, -1.0],
                 [-2.0, -1.5, 0.8],
                 [1.0, 0.2, 0.4]])
Y_ml = np.array([[1, 0, 0],
                 [1, 1, 0],
                 [0, 0, 1],
                 [1, 1, 0]])

### Генератор мини-батчей · база

В PyTorch данные подаются порциями через `DataLoader`. Напишите его упрощённый аналог — функцию-генератор `batches(X, y, size, rng)`, которая перемешивает номера строк (`rng.permutation`) и по очереди отдаёт пары `(X_batch, y_batch)` через `yield`. Проверьте на `X` из 10 строк с `size=4`: сохраните `sizes` — список размеров батчей и `covered` — `True`, если каждая строка попала ровно в один батч.

*Подсказки (2): `dsc.hint('batches')`*

In [ ]:
%%task ch11/04b-softmax/batches
import numpy as np
X = np.arange(20).reshape(10, 2)
y = np.arange(10)

### Батчи против полного спуска · посложнее

Обучите сеть из урока «Обучение сети: обратное распространение ошибки» (полумесяцы, 16 нейронов, `lr = 0.5`, веса из `np.random.default_rng(0)` как в заготовке) мини-батчами: 100 эпох, генератор `batches` из заготовки с `rng = np.random.default_rng(1)`, размеры батча 400, 64 и 16. Для каждого размера сохраните в словарь `res` тройку `(потеря на всех данных, доля верных, число шагов)` после обучения (потеря и доля — 3 знака). В `best_size` — размер с наименьшей потерей.

*Подсказки (3): `dsc.hint('ch11/04b-softmax/minibatch')`*

In [ ]:
%%task ch11/04b-softmax/minibatch
import numpy as np

def train_batches(size, epochs=100, lr=0.5):
    W1, b1, w2, b2 = init()
    rng = np.random.default_rng(1)
    steps = 0
    ...
    return W1, b1, w2, b2, steps

res = {}

### Многоклассовая сеть на numpy · со звёздочкой

Обучите на numpy сеть «2 → 16 (ReLU) → 3 (softmax)» на трёх классах (`make_blobs` из заготовки). Градиент кросс-энтропии по логитам — `(P - Y) / n`, где `Y` — one-hot; дальше обратный проход — как в уроке «Обучение сети: обратное распространение ошибки», только выход — матрица. Инициализация: `rng = np.random.default_rng(0)`, `W1 = rng.normal(size=(2, 16)) * 0.5`, `W2 = rng.normal(size=(16, 3)) * 0.5`, смещения — нули. Шаг 0.5, 500 шагов. Сохраните `acc` — долю правильных ответов на тех же данных (3 знака).

*Подсказки (2): `dsc.hint('softmax-net')` · задача по желанию*

In [ ]:
%%task ch11/04b-softmax/softmax-net
import numpy as np
from sklearn.datasets import make_blobs
X, y = make_blobs(300, centers=3, cluster_std=1.5, random_state=0)
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = np.eye(3)[y]

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

## Урок 7. Dropout, BatchNorm и smoke-test

Три приёма, без которых не обходится настоящая сеть: dropout против переобучения, BatchNorm — нормировка внутри сети, и smoke-test — минутная проверка, которая ловит ошибку в цикле обучения до многочасового запуска на видеокарте.

Теория — на сайте курса: `ch11/04c-debug.html`.

### Dropout руками · база

Напишите функцию `dropout(H, rate, rng)`: маска «оставить» — `rng.random(H.shape) >= rate`, результат — `H * mask / (1 - rate)`. Примените к матрице из единиц 1000 × 50 с `rate = 0.3` и генератором `np.random.default_rng(0)`. Сохраните `kept_share` — долю ненулевых элементов (3 знака) и `mean_after` — среднее результата (3 знака).

*Подсказки (2): `dsc.hint('dropout')`*

In [ ]:
%%task ch11/04c-debug/dropout
import numpy as np
H = np.ones((1000, 50))

### BatchNorm руками · база

Напишите две функции, как в теории: `batchnorm_train(Xb, gamma, beta, run_mean, run_var, momentum=0.1, eps=1e-5)` возвращает `(out, run_mean, run_var)` — нормированный батч и обновлённые скользящие средние (дисперсию считайте `Xb.var(axis=0)`); `batchnorm_eval(X, gamma, beta, run_mean, run_var, eps=1e-5)` нормирует по скользящим средним.

Данные — первые 6 гостей, три признака (`B` в заготовке), `gamma` и `beta` заданы, стартовые `run_mean` — нули, `run_var` — единицы. Пропустите через `batchnorm_train` сначала батч `B[:3]`, затем `B[3:]`. Сохраните `out1` — выход первого батча (3 знака), `run_mean` и `run_var` после двух батчей (4 знака) и `pred` — `batchnorm_eval` для первого гостя `B[:1]` после двух батчей (3 знака).

*Подсказки (2): `dsc.hint('batchnorm')`*

In [ ]:
%%task ch11/04c-debug/batchnorm
import numpy as np
import pandas as pd
g = pd.read_csv("guests_churn.csv")
B = g[["visits_90d", "days_since_last", "complaints"]].head(6).to_numpy(float)
gamma = np.array([1.0, 2.0, 0.5])
beta = np.array([0.0, 1.0, -1.0])
run_mean, run_var = np.zeros(3), np.ones(3)

### Smoke-test находит ошибку · посложнее

Коллега прислал две версии шага обучения сети с полумесяцами из урока «Обучение сети: обратное распространение ошибки» — `step_a` и `step_b` (они уже загружены, код скрыт). Обе выглядят рабочими: на полумесяцах потеря у обеих падает. Найдите версию с ошибкой smoke-тестом. В заготовке есть данные `X`, `y`, функции `init()` и `loss_of(params, X, y)` из теории.

Напишите `smoke(step, n=16, steps=300, lr=0.5)`: берёт свежие веса `init()`, считает стартовую потерю на **всех** `X, y`, затем делает `steps` шагов `params = step(params, X[:n], y[:n], lr)` только на первых `n` объектах и возвращает `(стартовая потеря, потеря на этих n объектах после обучения)`. Сохраните:

- `res` — словарь `{"a": …, "b": …}` с парами для обеих версий (3 знака);
- `start_ok` — `True`, если стартовая потеря отличается от $\ln 2$ меньше чем на 0.1;
- `buggy` — `"a"` или `"b"`: у какой версии потеря на 16 объектах не опустилась ниже 0.1;
- `both_learn` — `True`, если у обеих версий потеря на 16 объектах после обучения меньше стартовой.

*Подсказки (3): `dsc.hint('smoke-test')`*

In [ ]:
%%task ch11/04c-debug/smoke-test
import numpy as np

def smoke(step, n=16, steps=300, lr=0.5):
    params = init()
    ...

## Урок 8. Проект: распознать цифры в бумажных анкетах

Финал главы. Гости до сих пор заполняют бумажные анкеты программы лояльности, и операторы вручную перебивают номера телефонов. Пробуем распознавать рукописные цифры: от логистической регрессии до своей нейросети на numpy и MLP из sklearn — и находим слабое место полносвязных сетей, которое исправят свёрточные сети в главе 14.

Теория — на сайте курса: `ch11/05-project.html`.

### Шаг 1. Классы и «средние» цифры · база

Сохраните `counts` — число картинок каждой цифры (массив длины 10), `balanced` — `True`, если самый редкий класс встречается не реже чем в 0.9 раза от самого частого. Нарисуйте «среднюю» картинку каждой цифры (среднее `digits.images` по картинкам этой цифры) — 10 изображений в ряд, заголовок каждого — цифра. Сохраните `mean_imgs` — массив формы (10, 8, 8).

*Подсказки (2): `dsc.hint('ch11/05-project/look')`*

In [ ]:
%%task ch11/05-project/look
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
digits = load_digits()
y = digits.target

### Шаг 2–3. Простые модели · база

Разделите данные (`X = digits.data / 16`) и сравните на тесте `LogisticRegression(max_iter=2000)` и `KNeighborsClassifier(n_neighbors=3)`. Сохраните `acc_lr`, `acc_knn` (3 знака).

*Подсказки (2): `dsc.hint('simple')`*

In [ ]:
%%task ch11/05-project/simple
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

digits = load_digits()
X, y = digits.data / 16, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Шаг 4. Своя сеть на numpy · база

Обучите сеть «64 → 32 (ReLU) → 10 (softmax)» на numpy: `rng = np.random.default_rng(0)`, `W1 = rng.normal(size=(64, 32)) * 0.1`, затем `W2 = rng.normal(size=(32, 10)) * 0.1`, смещения — нули, шаг 0.5, 300 шагов по всей обучающей выборке. Сохраните `acc_own` — долю правильных ответов на тесте (3 знака).

Цикл — как в уроке «Обучение сети: обратное распространение ошибки», отличается только выход:

- прямой проход: `Z1 = X_train @ W1 + b1`, `H = np.maximum(0, Z1)`, `P = softmax(H @ W2 + b2)` — матрица вероятностей «картинки × 10 цифр»;
- ошибка на выходе: `D = (P - Y_train) / n`, где `Y_train` — one-hot-ответы (в заготовке), `n` — число обучающих картинок. Для softmax с кросс-энтропией это та же формула «p − y», только для матрицы;
- ошибка скрытого слоя: `dH = D @ W2.T * (Z1 > 0)` — `D @ W2.T` заменяет `np.outer(d2, w2)`, потому что выходов теперь 10;
- градиенты: `H.T @ D` для `W2`, `D.sum(axis=0)` для `b2`, `X_train.T @ dH` для `W1`, `dH.sum(axis=0)` для `b1`.

Если вы решали задачу «Многоклассовая сеть на numpy» урока «Softmax, выход под задачу и мини-батчи», это тот же код с другими размерами.

*Подсказки (2): `dsc.hint('own-net')`*

In [ ]:
%%task ch11/05-project/own-net
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X, y = digits.data / 16, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
Y_train = np.eye(10)[y_train]

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

### Шаг 5. MLP из sklearn · база

Обучите `MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=500, random_state=0)` (признаки уже в диапазоне 0–1, масштаб не нужен) и сохраните `acc_mlp` (3 знака) и `proba` — матрицу вероятностей на тесте (`predict_proba`).

*Подсказки (2): `dsc.hint('mlp')`*

In [ ]:
%%task ch11/05-project/mlp
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X, y = digits.data / 16, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Шаг 6. Что путается и что показать оператору · посложнее

Для MLP из прошлого шага (в заготовке) постройте матрицу ошибок `cm = confusion_matrix(y_test, pred)`. Сохраните `top_pair` — пару `(настоящая цифра, предсказанная)` с наибольшим числом ошибок (кортеж из двух int; при равенстве — первая в порядке строк и столбцов) и `n_errors` — всего ошибок. Затем правило для оператора: ответы с максимальной вероятностью меньше 0.9 отправляются на ручную проверку. Сохраните `review_share` — долю таких ответов и `errors_caught` — долю всех ошибок модели, которые попали на проверку (3 знака).

*Подсказки (2): `dsc.hint('ch11/05-project/errors')`*

In [ ]:
%%task ch11/05-project/errors
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

digits = load_digits()
X, y = digits.data / 16, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=500, random_state=0).fit(X_train, y_train)
proba = model.predict_proba(X_test)
pred = proba.argmax(axis=1)

### Шаг 7. Сдвиг на один пиксель · со звёздочкой

Убедитесь сами, что будет со сдвигом: сдвиньте тестовые картинки на 1 пиксель вправо (`np.roll(..., 1, axis=2)` по массиву формы (n, 8, 8), освободившийся левый столбец заполните нулями) и посчитайте долю правильных ответов MLP на сдвинутых картинках (`acc_shift`, 3 знака). Затем обучите тот же MLP на обучающих картинках **вместе** с их сдвинутыми копиями (вправо на 1 пиксель) и проверьте на сдвинутом тесте (`acc_aug`, 3 знака).

*Подсказки (2): `dsc.hint('ch11/05-project/shift')` · задача по желанию*

In [ ]:
%%task ch11/05-project/shift
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X, y = digits.data / 16, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=500, random_state=0).fit(X_train, y_train)

def shift_right(X):
    imgs = np.roll(X.reshape(-1, 8, 8), 1, axis=2)
    imgs[:, :, 0] = 0
    return imgs.reshape(-1, 64)

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()